# 🏷️ Notebook 03 — Medical Category Labelling
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Load the cleaned dataset (`pubmedqa_cleaned.csv`)
- Apply keyword-regex labelling to assign one of 6 medical categories
- Validate all 6 categories are present with ≥ 1% representation each
- Save the labelled dataset as `pubmedqa_labelled.csv`

### 📋 Output
- `data/processed/pubmedqa_labelled.csv` — dataset with `category` column

### 📌 Category Definitions
| Category | Description |
|---|---|
| Symptoms | Questions about signs, pain, manifestations |
| Diagnosis | Questions about detection, screening, assessment |
| Treatment | Questions about therapies, surgeries, procedures |
| Medication | Questions about drugs, dosages, vaccines |
| Prevention | Questions about risk reduction, lifestyle, avoidance |
| General | Everything else |

---

## 1. Imports & Configuration

In [1]:
import pandas as pd
import os
import sys
from pathlib import Path

# Add project root to path so we can import src modules
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.labeller import label_dataframe, print_category_distribution

print('✅ Imports successful')

✅ Imports successful


## 2. Load Cleaned Dataset

In [2]:
input_path = '../data/processed/pubmedqa_cleaned.csv'

df = pd.read_csv(input_path)

print(f'✅ Loaded: {input_path}')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Columns: {df.columns.tolist()}')

# Verify expected columns exist
assert 'question' in df.columns, "Missing 'question' column"
assert 'context' in df.columns, "Missing 'context' column"
assert 'answer' in df.columns, "Missing 'answer' column"

df.head(3)

✅ Loaded: ../data/processed/pubmedqa_cleaned.csv
   Shape: 211,188 rows × 3 columns
   Columns: ['question', 'context', 'answer']


,question,context,answer
0,are group 2 innate lymphoid cells ( ilc2s ) in...,chronic rhinosinusitis (crs) is a heterogeneou...,"as ilc2s are elevated in patients with crswnp,..."
1,does vagus nerve contribute to the development...,phosphatidylethanolamine n-methyltransferase (...,neuronal signals via the hepatic vagus nerve c...
2,does psammaplin a induce sirtuin 1-dependent a...,psammaplin a (psa) is a natural product isolat...,psa significantly inhibited mcf-7/adr cells pr...


## 3. Apply Category Labelling

We use `src/data/labeller.py` which applies keyword-regex matching
to the combined `question + context` text for each row.

In [3]:
print('⏳ Applying category labels...')

df = label_dataframe(
    df,
    question_col='question',
    context_col='context',
    output_col='category'
)

print('✅ Labelling complete!')
print(f'   New columns: {df.columns.tolist()}')

⏳ Applying category labels...
✅ Labelling complete!
   New columns: ['question', 'context', 'answer', 'category']


## 4. Validate Category Distribution

In [4]:
counts = print_category_distribution(df, col='category')


Category          Count Percentage
───────────────────────────────────
Medication       71,539      33.9%
Treatment        48,580      23.0%
Diagnosis        31,919      15.1%
General          28,252      13.4%
Prevention       22,179      10.5%
Symptoms          8,719       4.1%

Total rows: 211,188
Categories: 6

All categories have >= 1% representation


## 5. KPI Check — All 6 Categories ≥ 1%

In [5]:
total = len(df)
n_categories = df['category'].nunique()
min_pct = (df['category'].value_counts() / total * 100).min()

print(f'\n📊 KPI Validation:')
print(f'   Categories found: {n_categories} / 6 expected')
print(f'   Minimum category %: {min_pct:.2f}%')

assert n_categories == 6, f"Expected 6 categories, found {n_categories}"
assert min_pct >= 1.0, f"Category below 1%: {min_pct:.2f}%"

print('   ✅ All 6 categories present with ≥ 1% representation')


📊 KPI Validation:
   Categories found: 6 / 6 expected
   Minimum category %: 4.13%
   ✅ All 6 categories present with ≥ 1% representation


In [6]:
# Add this cell BEFORE the labelling step
import importlib
import src.data.labeller as labeller_module

# Force Python to reload the module (in case it cached the old version)
importlib.reload(labeller_module)

from src.data.labeller import CATEGORY_KEYWORDS

print("Prevention keywords:")
print(CATEGORY_KEYWORDS['Prevention'])
print(f"\nCount: {len(CATEGORY_KEYWORDS['Prevention'])}")

total = len(df)
for cat, count in df['category'].value_counts().items():
    print(f"{cat:<15} {count:>6} → {count/total*100:.2f}%")

Prevention keywords:
['prevent', 'prophyla', 'avoid', 'risk reduction', 'risk factor', 'early detection', 'screening program', 'health promotion', 'public health', 'epidemiolog', 'lifestyle', 'diet', 'exercise', 'smoking cessation', 'obesity', 'weight loss', 'physical activity', 'breastfeed', 'safe sex', 'hygiene', 'nutriti']

Count: 21
Medication       71539 → 33.87%
Treatment        48580 → 23.00%
Diagnosis        31919 → 15.11%
General          28252 → 13.38%
Prevention       22179 → 10.50%
Symptoms          8719 → 4.13%


## 6. Sample Rows per Category

In [7]:
pd.set_option('display.max_colwidth', 80)

for cat in sorted(df['category'].unique()):
    print(f'\n{"=" * 80}')
    print(f'Category: {cat}')
    print(f'{"=" * 80}')
    sample = df[df['category'] == cat].head(2)
    for _, row in sample.iterrows():
        print(f'  Q: {row["question"][:100]}')
        print(f'  A: {row["answer"][:100]}')
        print()


Category: Diagnosis
  Q: are higher nitric oxide levels associated with disease activity in egyptian rheumatoid arthritis pat
  A: there are increased levels of nitric oxide in the serum of patients with rheumatoid arthritis. nitri

  Q: is decreased expression of liver-type fatty acid-binding protein associated with poor prognosis in h
  A: l-fabp was downregulated in hcc and could be served as a promising prognostic marker for hcc patient


Category: General
  Q: is methylation of the fgfr2 gene associated with high birth weight centile in humans?
  A: we identified a novel biologically plausible candidate (fgfr2) for with bwc that merits further stud

  Q: do large portion sizes increase bite size and eating rate in overweight women?
  A: increasing portion size led to a larger bite size and faster eating rate, but a slower reduction in 


Category: Medication
  Q: does psammaplin a induce sirtuin 1-dependent autophagic cell death in doxorubicin-resistant mcf-7/ad
  A: psa signific

## 7. Save Labelled Dataset

In [8]:
output_path = '../data/processed/pubmedqa_labelled.csv'
os.makedirs('../data/processed', exist_ok=True)

# Verify final column set
expected_cols = ['question', 'context', 'answer', 'category']
assert list(df.columns) == expected_cols, \
    f"Expected {expected_cols}, got {list(df.columns)}"

df.to_csv(output_path, index=False)

print(f'✅ Labelled dataset saved to: {output_path}')
print(f'   Rows: {df.shape[0]:,}')
print(f'   Columns: {df.columns.tolist()}')
print(f'   File size: {os.path.getsize(output_path) / 1024**2:.1f} MB')

✅ Labelled dataset saved to: ../data/processed/pubmedqa_labelled.csv
   Rows: 211,188
   Columns: ['question', 'context', 'answer', 'category']
   File size: 351.5 MB


In [9]:
# ── Optional: sync labelled data to HuggingFace Hub ───────────────────────
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

if not os.getenv('HF_TOKEN'):
    print("⚠️  HF_TOKEN not set — skipping upload.")
    print("   Add it to your .env file (see .env.example).")
else:
    from src.data.hub import upload_file
    print("\n📤 Syncing to HuggingFace...")
    upload_file("data/processed/pubmedqa_labelled.csv", "processed/pubmedqa_labelled.csv")


📤 Syncing to HuggingFace...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  ✅ Uploaded: data/processed/pubmedqa_labelled.csv (351.5 MB)


## ✅ Summary

| Item | Result |
|---|---|
| Input | `data/processed/pubmedqa_cleaned.csv` |
| Output | `data/processed/pubmedqa_labelled.csv` |
| New column | `category` |
| Categories | Symptoms, Diagnosis, Treatment, Medication, Prevention, General |
| All ≥ 1% | ✅ |

---

### ➡️ Next Step
Open **`04_eda.ipynb`** to explore and visualise the labelled dataset.